# TelecomTS acquisition: data analysis and visualization

Inspect the outputs of issue #21: scenarios, analysis CSVs, conversion counts, and source uncertainty. This notebook does not download data, create splits, train models, approve text, or infer causes. Targets are displayed for dataset inspection only. Successful conversion does not establish independent train/test groups.

Install `uv sync --extra dev --extra analysis`, prepare data with the ingestion CLI, then run this notebook. Set `TELECOMTS_SCENARIO_DIR` to inspect another generated directory. The default is `data/scenarios/`. Figures are written under `artifacts/telecomts_eda/` (override with `TELECOMTS_EDA_DIR`).

In [ ]:
import json
import os
import sys
from pathlib import Path

%matplotlib inline
import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists() and (p / "src/clauseguard").is_dir())
sys.path.insert(0, str(ROOT / "src"))
from clauseguard.datasets.telecomts import file_sha256

DATA = Path(os.environ.get("TELECOMTS_SCENARIO_DIR", ROOT / "data/scenarios"))
FIGURES = Path(os.environ.get("TELECOMTS_EDA_DIR", ROOT / "artifacts/telecomts_eda"))
FIGURES.mkdir(parents=True, exist_ok=True)
if not (DATA / "csv_schema.json").exists():
    raise FileNotFoundError(f"Prepare analysis CSVs first; missing {DATA / 'csv_schema.json'}")
schema = json.loads((DATA / "csv_schema.json").read_text())
for name, expected_hash in schema["files"].items():
    assert file_sha256(DATA / name) == expected_hash, f"Changed CSV: {name}"
assert file_sha256(DATA / "scenarios.jsonl") == schema["source_scenarios_sha256"], "CSV/source mismatch"
overview = pd.read_csv(DATA / "scenarios_overview.csv", dtype=str, keep_default_na=False)
assert len(overview) == schema["scenarios"]
if overview.empty:
    raise ValueError("No accepted scenarios available; inspect rejected.jsonl and conversion.json")
conversion = json.loads((DATA / "conversion.json").read_text()) if (DATA / "conversion.json").exists() else {}

def show_plot(name):
    plt.tight_layout()
    plt.savefig(FIGURES / f"{name}.png", dpi=130, bbox_inches="tight")
    plt.show()

display(pd.DataFrame([conversion.get("counts", {"accepted": len(overview)})]))
print("Source hash:", schema["source_scenarios_sha256"])
print("Splits are not created or verified by this notebook.")

## Origin and category coverage

Counts use every accepted scenario in the selected acquisition. Normal and unknown labels are shown explicitly. Real faults in this release are controlled jamming; synthetic category accuracy would not establish general real-network RCA.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
overview["origin"].value_counts().sort_index().plot.bar(ax=axes[0], title="Scenario origin")
overview["rca_label"].value_counts().sort_values().plot.barh(ax=axes[1], title="Source category coverage")
axes[0].set_ylabel("Scenarios")
axes[1].set_xlabel("Scenarios")
show_plot("origin_and_categories")
display(pd.crosstab(overview["origin"], overview["rca_label"]))

## Window lengths, cadence, channel types, and units

Source timestamps remain source strings: no timezone is inferred. Unknown units are shown as unknown rather than being guessed from names or generated descriptions.

In [ ]:
channel_info = pd.DataFrame.from_dict(schema["channels"], orient="index")
channel_info["unit"] = channel_info["unit"].fillna("unknown")
display(channel_info)
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
overview["sample_count"].value_counts().sort_index().plot.bar(ax=axes[0], title="Window sample counts")
overview["sampling_rate_hz"].value_counts().sort_index().plot.bar(ax=axes[1], title="Sampling rates (Hz)")
show_plot("lengths_and_cadence")

## Missing values across the entire selected export

Scan CSV chunks to bound memory. Missing masks preserve the distinction between a real null and categorical strings such as `None` or an empty string. The distribution and correlation plots below use a small, deterministic illustrative selection, not the whole dataset.

In [ ]:
names = sorted(schema["channels"])
mask_columns = [name + "__missing" for name in names]
selected_ids = set()
for _, group in overview.groupby(["origin", "rca_label"], sort=True):
    selected_ids.update(group.sample(n=min(3, len(group)), random_state=42)["scenario_id"])
missing = pd.Series(0, index=mask_columns, dtype="int64")
observed_samples = 0
pieces = []
columns = ["scenario_id", "sample_index", "relative_time_seconds", *names, *mask_columns]
for chunk in pd.read_csv(DATA / "kpi_samples.csv", usecols=columns, dtype=str, keep_default_na=False, chunksize=50000):
    observed_samples += len(chunk)
    missing += chunk[mask_columns].eq("True").sum()
    selected = chunk.loc[chunk["scenario_id"].isin(selected_ids)]
    if not selected.empty:
        pieces.append(selected)
assert observed_samples == schema["samples"]
missing.index = names
(missing / observed_samples * 100).sort_values().plot.barh(figsize=(9, 5), title="Missing values across all exported samples (%)")
show_plot("missing_values")
sample = pd.concat(pieces, ignore_index=True)
sample["sample_index"] = pd.to_numeric(sample["sample_index"])
sample["relative_time_seconds"] = pd.to_numeric(sample["relative_time_seconds"])
numeric = [name for name in names if schema["channels"][name]["kind"] == "numeric"]
for name in numeric:
    sample[name] = pd.to_numeric(sample[name].where(sample[name + "__missing"].eq("False")), errors="raise")
print(f"Illustrative selection: {len(selected_ids)} windows, {len(sample)} samples; full export: {observed_samples} samples.")

## Numeric distributions and correlations in the illustrative selection

Channel scales differ, so each distribution uses its own axes. Correlation describes association in selected observations and does not prove a fault mechanism.

In [ ]:
if numeric:
    sample[numeric].hist(bins=25, figsize=(14, 3 * ((len(numeric) + 3) // 4)), layout=(-1, 4))
    show_plot("illustrative_distributions")
    correlations = sample[numeric].corr()
    fig, ax = plt.subplots(figsize=(10, 8))
    plot = ax.imshow(correlations.to_numpy(), vmin=-1, vmax=1, cmap="coolwarm")
    ax.set_xticks(range(len(numeric)), numeric, rotation=90)
    ax.set_yticks(range(len(numeric)), numeric)
    ax.set_title("Illustrative cross-channel correlations")
    fig.colorbar(plot, ax=ax, label="Pearson correlation")
    show_plot("illustrative_correlations")

## Representative KPI traces

Display one selected window per available origin. Axes use relative seconds, and source units remain explicit. Labels identify source annotations; plotted traces are not causal evidence or detection performance.

In [ ]:
chosen_channels = [name for name in ["RX_Bytes", "UL_BLER", "UL_SNR"] if name in numeric] or numeric[:3]
representatives = overview.loc[overview["scenario_id"].isin(selected_ids)].groupby("origin", sort=True).head(1)
if chosen_channels:
    fig, axes = plt.subplots(len(representatives), len(chosen_channels), figsize=(5 * len(chosen_channels), 3 * len(representatives)), squeeze=False)
    for row_index, (_, scenario) in enumerate(representatives.iterrows()):
        trace = sample.loc[sample["scenario_id"].eq(scenario["scenario_id"])].sort_values("sample_index")
        for column_index, name in enumerate(chosen_channels):
            ax = axes[row_index, column_index]
            ax.plot(trace["relative_time_seconds"], trace[name])
            ax.set_title(f"{scenario['origin']}: {name}")
            ax.set_xlabel("Relative seconds")
            ax.set_ylabel(schema["channels"][name]["unit"] or "Unit unknown")
    show_plot("representative_traces")

## Text review and recording relationships

Pending text is retained as reference data but does not enter model inputs. Unresolved relationships remain visible and require issue #22 work before scored train/test experiments. Duplicate observation matches are leads for review, not automatically verified shared recordings.

In [ ]:
display(pd.crosstab(overview["origin"], overview["text_review_status"]))
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
overview["text_review_status"].value_counts().plot.bar(ax=axes[0], title="Human text review status")
overview["relationship_status"].value_counts().plot.bar(ax=axes[1], title="Recording relationship status")
show_plot("text_and_lineage_coverage")
print("Exact observation duplicate matches:", len(conversion.get("duplicates", [])))
print("Approved nonempty text:", overview["has_approved_text"].eq("True").sum())
print("Figure directory:", FIGURES)
print("Conversion is not experiment readiness: grouping, splits and safe inference exports remain in issue #22.")